# Document Graph: Full Ingestion, Summarization & Navigation Pipeline

This notebook demonstrates the end-to-end **Document Graph pipeline** in `genai-graph` and `genai-tk`:

1. **Document Conversion & Table/Image Processing**:
   - Converting PDFs / documents to Markdown (`MarkItDownConverter`, `MistralOCRConverter`).
   - Lossless HTML table detection and conversion to standard Markdown tables (`is_markdown_table`, `convert_html_table`).
   - Image extraction and description caching with KV-store.
2. **Structured Outline & Summarization (BAML / LLM)**:
   - Table condensation for prompt token efficiency (> 30 lines).
   - Enriching sections with routing `description`, `summary`, and search `keywords`.
3. **Graph Construction & Ingestion (Ladybug DB)**:
   - Schema: `Folder ──CONTAINS──▶ Document ──HAS_SECTION──▶ MarkdownSection ──HAS_CHUNK──▶ SectionChunk`.
   - Building and ingesting via `DocumentGraphFactory` and `ingest_document_graph`.
4. **Agent Navigation & Hybrid Search**:
   - TOC tree generation with keywords and section summaries.
   - Section content retrieval with line-range pagination.
   - Hybrid BM25 + Vector search over section text and keywords.
   - Budget-constrained visual VLM image queries (`query_image`).

## 1. Setup & Environment

We set up our workspace directories and configure a temporary Ladybug graph database.

In [1]:
import asyncio
import tempfile
from pathlib import Path

# Sample PDF in tests/data
SAMPLE_PDF = Path("../tests/data/sample-pdf-a4-size.pdf").resolve()
if not SAMPLE_PDF.exists():
    SAMPLE_PDF = Path("tests/data/sample-pdf-a4-size.pdf").resolve()

WORK_DIR = Path(tempfile.mkdtemp())
CORPUS_DIR = WORK_DIR / "corpus"
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK_DIR / "docgraph.db"

print(f"Sample PDF : {SAMPLE_PDF} (exists: {SAMPLE_PDF.exists()})")
print(f"Corpus Dir : {CORPUS_DIR}")
print(f"Graph DB   : {DB_PATH}")

Sample PDF : /home/tcl/prj/genai-graph/tests/data/sample-pdf-a4-size.pdf (exists: True)
Corpus Dir : /tmp/tmp_1q6sqz6/corpus
Graph DB   : /tmp/tmp_1q6sqz6/docgraph.db


## 2. Document Conversion & HTML Table / Image Processing

### Lossless HTML Table Conversion
`genai_tk.extra.markdownize.table_processor` inspects HTML tables:
- If simple (no `rowspan`, `colspan`, or nested tables), converts losslessly to Markdown tables to save tokens.
- If complex, preserves the HTML table structure and annotates with `<!-- Table: {rows}x{cols} -->`.

In [2]:
from genai_tk.extra.markdownize.table_processor import (
    is_markdown_table,
    convert_html_table,
    get_table_dimensions,
    process_markdown_tables,
)

# 1. Simple table -> convertible to Markdown
simple_html = """
<table>
  <thead><tr><th>Metric</th><th>2024</th><th>2025</th></tr></thead>
  <tbody><tr><td>Revenue</td><td>$100M</td><td>$130M</td></tr></tbody>
</table>
"""
print(f"Simple table is_markdown_table: {is_markdown_table(simple_html)}")
print("Converted to Markdown:")
print(convert_html_table(simple_html))
print("-" * 40)

# 2. Complex table with colspan -> kept as HTML with dimension comments
complex_html = """
<table>
  <tr><th colspan="2">Financial Summary</th></tr>
  <tr><td>Assets</td><td>$500M</td></tr>
</table>
"""
print(f"Complex table is_markdown_table: {is_markdown_table(complex_html)}")
print(convert_html_table(complex_html))

Simple table is_markdown_table: True
Converted to Markdown:
| Metric | 2024 | 2025 |
| --- | --- | --- |
| Revenue | $100M | $130M |
----------------------------------------
Complex table is_markdown_table: False
<!-- Table: 2x2 -->
<table>
<tr><th colspan="2">Financial Summary</th></tr>
<tr><td>Assets</td><td>$500M</td></tr>
</table>



### PDF Conversion to Markdown
We convert our sample PDF to Markdown and store it in our corpus directory.

In [3]:
from genai_tk.extra.markdownize.markitdown_converter import MarkItDownConverter

converter = MarkItDownConverter()
converted_md = await converter.convert(SAMPLE_PDF)

# Clean tables and normalize
processed_md = process_markdown_tables(converted_md)

md_doc_path = CORPUS_DIR / "sample_report.md"
md_doc_path.write_text(processed_md, encoding="utf-8")

print(f"Saved converted document ({len(processed_md)} chars) to {md_doc_path}")
print("\n--- Markdown Preview ---\n")
print("\n".join(processed_md.splitlines()[:25]))

Saved converted document (2382 chars) to /tmp/tmp_1q6sqz6/corpus/sample_report.md

--- Markdown Preview ---

Company, Inc.

Sample
PDF

Prepared By

Chadwick Hilarity

+123-456-7890

https://sample-files.com


Sample PDF Content

Introduction

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Integer
nec odio. Praesent libero. Sed cursus ante dapibus diam. Sed nisi.
Nulla quis sem at nibh elementum imperdiet.

Objectives

The main objectives of this document are:


## 3. Section Outline, Summarization & Keywords (BAML / LLM)

When `outline_config` is provided to `DocumentGraphFactory`, an LLM / BAML function produces a content-free outline:
- Section `description` (one sentence routing signal).
- Section `summary` (for substantial sections).
- Section `keywords` (3-7 search terms).
- Long tables (>30 lines) are automatically condensed with head/tail sampling when building the prompt.

In [ ]:
from genai_graph.kg.document_graph.outline_extract import (
    OutlineConfig,
    DocumentOutline,
    OutlineEntry,
    _clean_markdown_for_prompt,
)

# Demonstration of table condensation for LLM prompts
tall_table = "| Col A | Col B |\n|---|---|\n" + "\n".join(f"| Val {i} | Num {i * 10} |" for i in range(40))
condensed = _clean_markdown_for_prompt(tall_table, head_rows=3, tail_rows=2)
print("--- Condensed Table for LLM Prompt ---")
print(condensed)

--- Condensed Table for LLM Prompt ---

| Col A | Col B |
|---|---|
| Val 0 | Num 0 |
| Val 1 | Num 10 |
| Val 2 | Num 20 |
| ... (34 table rows omitted for brevity; columns: Col A, Col B) |
| Val 37 | Num 370 |
| Val 38 | Num 380 |
| Val 39 | Num 390 |


## 4. Build & Ingest Document Graph into Ladybug

We configure `DocumentGraphFactory` and run `ingest_document_graph` to populate our Ladybug database.

In [5]:
from genai_graph.kg.backend import KuzuBackend
from genai_graph.kg.document_graph.ingest import ingest_document_graph
from genai_graph.kg.factories.document_graph_factory import DocumentGraphFactory

# 1. Initialize backend
backend = KuzuBackend()
backend.connect(str(DB_PATH))

# 2. Factory configuration (can also be loaded from app_conf.yaml)
factory = DocumentGraphFactory(
    sources=[str(CORPUS_DIR)],
    include=["*.md"],
)

# Inspect schema
schema = factory.build_schema()
print("Node labels:", [n.label for n in schema.nodes])
print("Relationship labels:", [r.name for r in schema.relations])

# 3. Run Ingestion
result = ingest_document_graph(backend, factory)
print("\n--- Ingestion Result ---")
print(f"Documents processed : {result.documents_processed}")
print(f"Sections created    : {result.sections_created}")
print(f"Relationships       : {result.relationships_created}")

[BAML] Log level set to ERROR


2026-09-14 16:02:42.529 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:504 - Creating node table: CREATE NODE TABLE IF NOT EXISTS Folder(name STRING, _original_name STRING, _created_at STRING, _updated_at STRING, folder_id STRING, parent_folder_id STRING, uri STRING, kind STRING, PRIMARY KEY(folder_id))
2026-09-14 16:02:42.586 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:504 - Creating node table: CREATE NODE TABLE IF NOT EXISTS Document(name STRING, _original_name STRING, _created_at STRING, _updated_at STRING, content_hash STRING, markdown_hash STRING, filename STRING, folder_id STRING, relative_path STRING, path STRING, file_size INT64, mime_type STRING, modified_at STRING, token_count INT64, section_count INT64, description STRING, summary STRING, access_level STRING, allowed_roles STRING[], allowed_users STRING[], PRIMARY KEY(content_hash))
2026-09-14 16:02:42.605 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:504 - Creating node table: CREATE NODE 

Node labels: ['Folder', 'Document', 'MarkdownSection', 'SectionChunk']
Relationship labels: ['CONTAINS', 'HAS_SUBFOLDER', 'HAS_SECTION', 'HAS_SUBSECTION', 'HAS_CHUNK']


2026-09-14 16:02:42.828 | INFO     | genai_graph.kg.document_graph.ingest:ingest_document_graph:344 - Ingested [1/1]: sample_report.md (sections=6, chunks=0)
2026-09-14 16:02:42.829 | DEBUG    | genai_graph.kg.ingest.merge:merge_nodes_batch:694 - Merging 1 Folder nodes via Arrow table...
2026-09-14 16:02:42.867 | DEBUG    | genai_graph.kg.ingest.merge:merge_nodes_batch:787 -   Folder: 1 processed via batch merge
2026-09-14 16:02:42.868 | DEBUG    | genai_graph.kg.ingest.merge:merge_nodes_batch:694 - Merging 1 Document nodes via Arrow table...
2026-09-14 16:02:42.891 | DEBUG    | genai_graph.kg.ingest.merge:merge_nodes_batch:787 -   Document: 1 processed via batch merge
2026-09-14 16:02:42.894 | DEBUG    | genai_graph.kg.ingest.merge:merge_nodes_batch:694 - Merging 6 MarkdownSection nodes via Arrow table...
2026-09-14 16:02:42.914 | DEBUG    | genai_graph.kg.ingest.merge:merge_nodes_batch:787 -   MarkdownSection: 6 processed via batch merge
2026-09-14 16:02:42.916 | DEBUG    | genai_gra


--- Ingestion Result ---
Documents processed : 1
Sections created    : 6
Relationships       : 7


## 5. Navigation & TOC Exploration

The document graph provides powerful navigation functions that return structured YAML outlines and table of contents.

In [6]:
from genai_graph.kg.query.document_graph_tools import (
    list_documents,
    get_document_toc,
    document_toc_yaml,
    get_section_content,
)

# List documents
docs = list_documents(backend)
print("Ingested Documents:")
for d in docs:
    print(f" - [{d['content_hash'][:8]}] {d['filename']} ({d['section_count']} sections)")

# Hierarchical TOC as YAML
first_doc = docs[0]
toc_yaml_str = document_toc_yaml(backend, first_doc["content_hash"], include_summaries=True)
print("\n--- Document TOC (YAML) ---\n")
print(toc_yaml_str)

Ingested Documents:
 - [342993fd] sample_report.md (6 sections)

--- Document TOC (YAML) ---

document: sample_report.md
id: 342993fd7c37ac70
sections:
- id: 342993fd7c37ac70::1
  title: Item 1
- id: 342993fd7c37ac70::2
  title: Item 2
- id: 342993fd7c37ac70::3
  title: Item 3
- id: 342993fd7c37ac70::4
  title: Item 4
- id: 342993fd7c37ac70::5
  title: Item 5



### Reading Section Content with Line Pagination
`get_section_content` allows retrieving exact text slices without overflowing the model context.

In [ ]:
toc_rows = get_document_toc(backend, first_doc["content_hash"])
if len(toc_rows) > 1:
    target_section_id = toc_rows[1]["section_id"]
    print(f"Reading section: {target_section_id}")

    # Read lines 1 to 20
    content = get_section_content(backend, [target_section_id])
    print("\n--- Section Text ---\n")
    print(content[0]["text"][:400])

Reading section: 342993fd7c37ac70::1

--- Section Text ---

Item 1



## 6. Hybrid & BM25 Section Search

`search_sections` searches over section titles, body text, summaries, and extracted keywords using BM25 and vector fusion.

In [ ]:
from genai_graph.kg.query.document_graph_tools import search_sections

query = "Market Analysis"
hits = search_sections(backend, query=query, mode="bm25", limit=5)

print(f"Search Results for '{query}':")
for h in hits:
    print(f" - [{h['section_id']}] {h['title']} (line {h['line_start']}) — score: {h['score']}")
    if h.get("matched_chunk"):
        print(f"   Snippet: {h['matched_chunk']!r}")

2026-09-14 16:03:14.033 | DEBUG    | genai_graph.kg.query.document_graph_tools:_keyword_section_hits:884 - FTS search unavailable, falling back to CONTAINS: Binder exception: Table MarkdownSection doesn't have an index with name section_fts.


Search Results for 'Market Analysis':
 - [342993fd7c37ac70::0] (document root) (line 1) — score: 2
 - [342993fd7c37ac70::5] Item 5 (line 157) — score: 1


## 7. LangChain Agent Navigation Tools & Image Query Budget

`create_document_graph_tools` packages the navigation API into standard LangChain `@tool` functions with a strict per-turn call limit for visual VLM queries (`max_image_queries=3`).

In [9]:
from genai_graph.kg.query.document_graph_tools import create_document_graph_tools

tools = create_document_graph_tools(str(DB_PATH), max_image_queries=3)

print(f"Available Agent Tools ({len(tools)}):")
for t in tools:
    print(f" • {t.name}: {t.description.splitlines()[0]}")

Available Agent Tools (6):
 • get_folder_toc: Start here. List the documents in a folder, each with an id and a one-line description.
 • get_document_toc: Get one document's section tree as YAML: each section's id, title, size and description.
 • get_section_content: Fetch the raw Markdown text of one or more sections. Comma-separated section_ids.
 • search_sections: Semantic + keyword search over section titles, text, and chunks, ranked by relevance.
 • query_image: Query or analyze an image using a Vision-Language Model (VLM).
 • list_documents: List every ingested document with its section count and one-line description.


## 8. Schema & Graph Interactive Visualization

In [15]:
from genai_graph.kg.schema import ResolvedSchema
from genai_graph.utils.notebooks import show_html_in_notebook

resolved = ResolvedSchema.from_graph_schema(schema)
schema_html = resolved.to_html()

show_html_in_notebook(schema_html, "docgraph_schema", height=450)
print(f"Schema Visualizer generated ({len(schema_html):,} bytes)")

Schema Visualizer generated (284,601 bytes)


## 9. Configuration-Driven Build (YAML & Pydantic)

In production or benchmark workflows, document graph builds can be orchestrated via YAML configuration or typed Pydantic models.

### YAML Schema (`app_conf.yaml` / `bench.yaml`)
```yaml
kg_build:
  llms:
    default: "glm_5.3_flash@openrouter"
  outline:
    structure_strategy: "auto"        # 'auto' | 'algo' | 'toc_preamble' | 'llm_full'
    generate_summaries: true          # Generate section descriptions, summaries & keywords
    max_description_words: 20
    max_summary_words: 60
    workers: 4
  retrieval:
    embeddings_id: "default"          # Enables SectionChunk embeddings + BM25 FTS
    chunk_size_tokens: 1500
    fts: true
```

### Pydantic Equivalent & Programmatic Invocation

In [13]:
from genai_graph.kg.document_graph.outline_extract import OutlineConfig
from genai_graph.kg.document_graph.retrieval import RetrievalConfig
from genai_graph.orchestration.document_graph_flow import document_graph_flow

# 1. Instantiate typed configuration
outline_cfg = OutlineConfig(
    structure_strategy="auto",
    generate_summaries=True,
    max_description_words=20,
    max_summary_words=60,
    workers=4,
)

retrieval_cfg = RetrievalConfig(
    embeddings_id=None,  # Set to an embedding model ID to enable vector chunk indexing
    chunk_size_tokens=1500,
    fts=True,
)

print("Outline Config:")
print(outline_cfg.model_dump_json(indent=2))

print("\nRetrieval Config:")
print(retrieval_cfg.model_dump_json(indent=2))

# 2. Programmatic flow execution (mirrors `cli docgraph build` and YAML workflow steps)
flow_result = document_graph_flow.fn(
    sources=[str(CORPUS_DIR)],
    db_path=str(DB_PATH),
    structure_strategy=outline_cfg.structure_strategy,
    generate_summaries=outline_cfg.generate_summaries,
    embeddings_id=retrieval_cfg.embeddings_id,
    fts=retrieval_cfg.fts,
    chunk_size_tokens=retrieval_cfg.chunk_size_tokens,
)

print(f"\nFlow Ingestion Outcome: {flow_result}")

2026-09-14 16:05:09.550 | INFO     | genai_graph.kg.factories.document_graph_factory:_get_files:181 - DocumentGraphFactory: discovered 1 file(s) from 1 source(s)
2026-09-14 16:05:09.555 | INFO     | genai_graph.kg.factories.document_graph_factory:extract_outlines:231 - Outline pre-pass: 1 file(s), 0 degraded, 0 LLM call(s)
2026-09-14 16:05:09.556 | INFO     | genai_graph.kg.document_graph.build:build_document_graph:247 - Outline pre-pass: 1 file(s), 0 degraded, 0 LLM call(s) in 0.0s
2026-09-14 16:05:09.557 | INFO     | genai_graph.kg.document_graph.build:build_document_graph:255 - Ingesting Document Graph from ['/tmp/tmp_1q6sqz6/corpus'] into /tmp/tmp_1q6sqz6/docgraph.db (llm=None, embeddings=None, fts=True)
2026-09-14 16:05:09.560 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:504 - Creating node table: CREATE NODE TABLE IF NOT EXISTS Folder(name STRING, _original_name STRING, _created_at STRING, _updated_at STRING, folder_id STRING, parent_folder_id STRING, uri STRING, kind

Outline Config:
{
  "llm": null,
  "structure_strategy": "auto",
  "generate_summaries": true,
  "workers": 4,
  "context_safety_ratio": 0.9,
  "summary_min_tokens": 800,
  "max_description_words": 20,
  "max_summary_words": 60,
  "max_description_chars": 180,
  "max_summary_chars": 500,
  "llm_max_tokens": null,
  "retry_max_tokens": 32000,
  "cache_root": null
}

Retrieval Config:
{
  "embeddings_id": null,
  "chunk_size_tokens": 1500,
  "fts": true
}

Flow Ingestion Outcome: {'db_path': '/tmp/tmp_1q6sqz6/docgraph.db', 'documents_processed': 1, 'documents_skipped': 1, 'documents_failed': 0, 'sections_created': 0, 'sections_summarized': 0, 'relationships_created': 0, 'files_degraded': 0, 'warnings': ['Could not create FTS index: Binder exception: Full text search index can only be built on string properties.']}
